# Lesson 23 - Instructor solutions

Attempt the learner notebook first. These are one set of possible solutions, with reasoning and
executable checks. Restart and run this notebook from the top: the worked examples below supply
the same definitions as the learner lesson. No learner answers or state are required.


## 1. Series and DataFrames add labels

A Series is a one-dimensional labeled column. A DataFrame is a table of columns, each with
its own dtype, sharing a row index. Labels make operations expressive but also cause alignment:
pandas aligns Series by labels rather than blindly by position. NumPy arrays are mainly positional.
Predict the missing result below. A new label does not automatically match by position.


In [ ]:
import pandas as pd
import numpy as np
from io import StringIO

monthly = pd.Series([10., 20.], index=["site-a", "site-b"], name="kwh")
adjustments = pd.Series([1.], index=["site-b"])
print(monthly + adjustments)
assert pd.isna((monthly + adjustments).loc["site-a"])
assert (monthly + adjustments).loc["site-b"] == 21


## 2. Load and inspect small CSV and JSON data

Reuse Lesson 10's file boundaries with pandas readers. StringIO acts like a text file so no
download or persistent file is needed. `read_csv` also accepts a local Path. `head`, `shape`,
`dtypes`, `info`, and `describe` help diagnose an unfamiliar table before transforming it.
Read numeric-looking source values as strings here so invalid entries remain visible.


In [ ]:
csv_text = """id,department,kwh
101, Finance ,10
102,IT,20
103,finance,bad
104,IT,
105,HR,30
106,HR,-5
"""
raw = pd.read_csv(StringIO(csv_text), dtype={"kwh": "string"})
print(raw.head(3))
print(raw.shape, raw.dtypes, sep="\n")
raw.info()
lookup = pd.read_json(StringIO('[{"department":"finance","manager":"Mira"},'
                               '{"department":"it","manager":"Dev"},'
                               '{"department":"hr","manager":"Tara"}]'))
assert raw.shape == (6, 3)


## 3. Select rows and columns explicitly

`df["column"]` returns a Series; `df[["column"]]` retains a DataFrame. `.loc` selects labels
and `.iloc` selects zero-based positions. Label slices include their stop; positional slices
exclude it. Combine masks with parentheses and `&`, not `and`. Use `.loc` for assignment,
not chained indexing such as `df[mask]["kwh"] = ...`.


In [ ]:
print(raw["id"].shape, raw[["id"]].shape)
print(raw.iloc[:2, :2])
indexed = raw.set_index("id")
print(indexed.loc[101:103, ["department", "kwh"]])
print(raw.loc[(raw["id"] >= 102) & (raw["id"] <= 104), ["id", "kwh"]])
assert indexed.loc[102, "department"] == "IT"


## 4. Clean types and retain rejection reasons

`to_numeric(errors="coerce")` turns malformed values into missing values; it does not decide
whether losing those values is acceptable. Count and report them. Normalize department text,
reject missing/negative/nonfinite energy, and keep accepted and rejected rows separately.
Do not replace unknown energy with zero unless zero is the documented meaning.


In [ ]:
cleaned = raw.copy()
cleaned["department"] = cleaned["department"].str.strip().str.casefold()
cleaned["kwh"] = pd.to_numeric(cleaned["kwh"], errors="coerce").astype(float)
valid_mask = np.isfinite(cleaned["kwh"]) & (cleaned["kwh"] >= 0)
rejected = cleaned.loc[~valid_mask].copy()
rejected["reason"] = np.where(rejected["kwh"].isna(), "missing or malformed", "negative or nonfinite")
accepted = cleaned.loc[valid_mask].copy()
print(rejected[["id", "reason"]])
assert accepted["id"].tolist() == [101,102,105]
assert rejected["id"].tolist() == [103,104,106]


## 5. Missing-value policies and new columns

`isna` detects missing values. `dropna(subset=...)` removes rows missing specified fields;
`fillna` supplies a documented replacement. pandas nullable `Int64` supports missing whole numbers.
Arithmetic between columns is vectorized. Keep units in names; rounding for display need not
alter stored numbers. Change the tariff and predict the total cost.


In [ ]:
optional = pd.Series(["2", None, "4"], dtype="string")
counts = pd.to_numeric(optional).astype("Int64")
print(counts, counts.fillna(0).tolist(), sep="\n")
print(cleaned.dropna(subset=["kwh"])["id"].tolist())  # Negative values still require a rule.
accepted["cost"] = accepted["kwh"] * 3
ranked = accepted.sort_values(["kwh", "id"], ascending=[False, True])
print(ranked[["id", "kwh", "cost"]])
assert accepted["cost"].sum() == 180


## 6. Group and aggregate

`groupby` partitions rows by a key. Named aggregations give readable result columns; `size`
counts rows, while `count` counts nonmissing values. `as_index=False` leaves grouping labels
as columns. Grouping is the table equivalent of the dictionaries and SQL summaries you used earlier.


In [ ]:
summary = accepted.groupby("department", as_index=False).agg(
    records=("id", "size"), total_kwh=("kwh", "sum"), mean_kwh=("kwh", "mean"))
print(summary)
print(accepted[["kwh", "cost"]].describe())
assert summary["total_kwh"].sum() == 60


## 7. Join with a cardinality contract

`merge` joins on column keys. A left join preserves the accepted readings even if a manager
is absent. `validate="many_to_one"` requires a unique lookup key; otherwise a duplicate manager
row could multiply measurements and totals. `indicator=True` reveals unmatched keys.
Check null join keys explicitly: pandas can match null keys to each other, unlike usual SQL expectations.


In [ ]:
joined = accepted.merge(lookup, on="department", how="left", validate="many_to_one", indicator=True)
print(joined[["id", "department", "manager", "_merge"]])
assert len(joined) == len(accepted)
assert joined["manager"].notna().all()
try:
    accepted.merge(pd.concat([lookup, lookup.iloc[:1]]), on="department", validate="many_to_one")
except pd.errors.MergeError:
    print("Duplicate lookup key rejected before totals can be multiplied.")
# JSON records are suitable for the next application's boundary.
print(accepted[["id", "kwh"]].to_json(orient="records"))


## 8. Common mistakes and debugging

- Inspect `shape`, `dtypes`, null counts, and a few rows after each boundary.
- A string column can sort `"100"` before `"20"`; convert before numerical comparisons.
- Use `.loc[mask, column]` to update, and `.copy()` for an intentionally separate subset.
- Do not trust a total after a join until row counts, key uniqueness, and unmatched rows are checked.
- Cleaning for exploratory analysis is different from ML imputation: learned statistics for ML must
come from training data only, as Lesson 24 explains.


## Exercise 1

A list of records maps naturally to rows; selecting one column produces a Series.


In [ ]:
practice_df = pd.DataFrame([{"id":1,"kwh":10},{"id":2,"kwh":20}])
assert practice_df.shape == (2,2)
assert practice_df["kwh"].sum() == 30


## Exercise 2

Masking chooses rows; sorting then changes their order without changing the selection rule.


In [ ]:
practice_df = pd.DataFrame({"id":[1,2,3],"kwh":[5,20,15]})
practice_ranked = practice_df.loc[practice_df["kwh"] >= 15].sort_values("kwh", ascending=False)
assert practice_ranked["id"].tolist() == [2,3]


## Exercise 3

A sum skips missing values by default, so report coverage alongside it.


In [ ]:
practice_n = pd.to_numeric(pd.Series(["10","bad",None,"0"]), errors="coerce")
assert practice_n.isna().sum() == 2
assert practice_n.sum() == 10
assert practice_n.iloc[3] == 0


## Exercise 4

Explicit names make the summary usable by reporting code.


In [ ]:
practice_df = pd.DataFrame({"department":["it","hr","it"],"kwh":[10,5,20]})
practice_grouped = practice_df.groupby("department").agg(count=("kwh","size"),total=("kwh","sum"))
assert practice_grouped.loc["it"].tolist() == [2,30]
assert practice_grouped.loc["hr"].tolist() == [1,5]


## Exercise 5

Preserve unmatched facts for review rather than silently dropping them with an inner join.


In [ ]:
practice_readings = pd.DataFrame({"department":["it","legal"],"kwh":[10,5]})
practice_lookup = pd.DataFrame({"department":["it"],"manager":["Dev"]})
practice_joined = practice_readings.merge(practice_lookup, on="department", how="left", validate="many_to_one", indicator=True)
assert len(practice_joined) == 2
assert practice_joined.loc[practice_joined["_merge"] == "left_only", "department"].tolist() == ["legal"]
try:
    practice_readings.merge(pd.concat([practice_lookup]*2), on="department", validate="many_to_one")
except pd.errors.MergeError:
    pass
else:
    raise AssertionError("Duplicate lookup accepted")


## Chapter project

Build `analyze_energy(csv_text, managers)` using the six-row CSV from section 2. Normalize
departments, convert energy, retain invalid rows with reasons, join managers with many-to-one
validation, and return a department summary plus rejected rows. Require nonblank departments and
unique nonmissing IDs; reject the whole input on duplicate/missing IDs. Do not mutate the lookup.

**Check:** 3 accepted and 3 rejected, total 60 kWh; department totals finance=10, it=20, hr=30.
Export summary to JSON records and read it back using StringIO. Duplicate lookup keys must fail.

The implementation separates inputs, transformation, and reporting.


In [ ]:
def analyze_energy(csv_text, managers):
    table = pd.read_csv(StringIO(csv_text), dtype={"kwh":"string"})
    if table["id"].isna().any() or table["id"].duplicated().any():
        raise ValueError("IDs must be present and unique")
    table["department"] = table["department"].astype("string").str.strip().str.casefold()
    table["kwh"] = pd.to_numeric(table["kwh"], errors="coerce").astype(float)
    valid = (np.isfinite(table["kwh"]) & (table["kwh"] >= 0)
             & table["department"].notna() & table["department"].ne(""))
    rejected = table.loc[~valid].copy()
    rejected["reason"] = "invalid energy or department"
    good = table.loc[valid].merge(managers, on="department", how="left", validate="many_to_one")
    summary = good.groupby("department", as_index=False).agg(records=("id","size"), total_kwh=("kwh","sum"))
    return summary, rejected
project_summary, project_rejected = analyze_energy(csv_text, lookup)
assert project_summary["records"].sum() == 3 and len(project_rejected) == 3
assert dict(zip(project_summary["department"],project_summary["total_kwh"])) == {"finance":10,"hr":30,"it":20}
project_loaded = pd.read_json(StringIO(project_summary.to_json(orient="records")))
assert project_loaded["total_kwh"].sum() == 60
try:
    analyze_energy(csv_text, pd.concat([lookup, lookup.iloc[:1]]))
except pd.errors.MergeError:
    pass
else:
    raise AssertionError("Duplicate lookup accepted")
